> **Note:** Before running this notebook, ensure Application Insights is configured for your Azure AI Project. If you haven't set it up yet, complete the setup exercise in [14-foundry-agent-monitor.ipynb](./14-foundry-agent-monitor.ipynb) first.

# Sequential Agent Workflow: Attraction Recommendation System

This notebook demonstrates a **sequential multi-agent workflow** using Microsoft Agent Framework with agents from Azure AI Foundry.

## Use Case: Travel Recommendation System

We'll create a two-agent system that works sequentially:

1. **Front Desk Agent**: Makes attraction recommendations based on user's city interest
2. **Concierge Agent**: Reviews and rates the recommendation with expert analysis

## Key Benefits of Sequential Orchestration

- **Iterative Refinement**: Second agent improves upon first agent's work
- **Specialization**: Each agent has a specific role
- **Quality Control**: Built-in review and validation
- **Clear Information Flow**: Structured handoff between agents

## Prerequisites

Before running this notebook, ensure you have:

- ✅ Microsoft Agent Framework installed (`pip install agent-framework`)
- ✅ Azure AI Foundry project configured
- ✅ Two agents created in Azure AI Foundry:
  - Agent named "**frontdeskagent**"
  - Agent named "**conciergeagent**"
- ✅ Azure CLI authenticated (`az login`)
- ✅ Environment variable: `AZURE_AI_PROJECT_ENDPOINT`
- ✅ (Optional) Application Insights for telemetry

## Step 1: Import Required Libraries

Import all necessary packages for the agent framework, Azure integration, and observability.

In [1]:
import os
import random
import time
from typing import Annotated

# Agent Framework
from agent_framework import ChatMessage, SequentialBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from agent_framework.observability import get_tracer

# Azure imports
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential

# Data modeling and display
from pydantic import BaseModel
from IPython.display import HTML, display
from dotenv import load_dotenv

# Telemetry
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

# Enable nested asyncio for Jupyter notebooks
import nest_asyncio
nest_asyncio.apply()

print("✅ All imports successful!")

✅ All imports successful!


## Step 2: Define Data Models

These Pydantic models define the structured output format for each agent.

In [2]:
class AttractionRecommendation(BaseModel):
    """Attraction recommendation from the front desk agent."""
    city: str
    attraction_name: str
    description: str
    category: str  # e.g., "museum", "landmark", "park"
    recommended_duration: str  # e.g., "2-3 hours"
    why_recommended: str
    best_time_to_visit: str


class AttractionReview(BaseModel):
    """Expert review and rating from the concierge agent."""
    attraction_name: str
    city: str
    popularity_score: int  # 1-10 scale
    popularity_reasoning: str
    visitor_rating: float  # 1.0-5.0 scale
    pros: list[str]
    cons: list[str]
    concierge_recommendation: str
    alternative_suggestions: list[str]

print("✅ Data models defined")

✅ Data models defined


## Step 3: Define Tool Functions

These tools provide real-time information to agents. They will be passed to agents when retrieving them from Azure AI Foundry.

### Front Desk Agent Tools
- Get attraction opening hours
- Check current weather
- Calculate distances

### Concierge Agent Tools
- Get visitor reviews
- Compare multiple attractions
- Get ticket pricing

In [3]:
# Front Desk Agent Tools
def get_attraction_hours(
    attraction_name: Annotated[str, "Name of the attraction"],
    city: Annotated[str, "City where the attraction is located"]
) -> str:
    """Get opening hours for a specific attraction."""
    print(f"🔧 TOOL CALLED: get_attraction_hours('{attraction_name}', '{city}')")
    hours = {"weekday": "9:00 AM - 6:00 PM", "weekend": "10:00 AM - 8:00 PM", "closed": "Tuesdays"}
    result = f"{attraction_name} in {city} is open {hours['weekday']} on weekdays, {hours['weekend']} on weekends. Closed on {hours['closed']}."
    print(f"   ↳ Returned: {result[:60]}...")
    return result

def get_current_weather(
    city: Annotated[str, "City name"],
    country: Annotated[str, "Country code"] = "US"
) -> str:
    """Get current weather conditions for a city."""
    print(f"🔧 TOOL CALLED: get_current_weather('{city}', '{country}')")
    conditions = ["Sunny", "Partly Cloudy", "Overcast", "Light Rain"]
    temp = random.randint(15, 28)
    result = f"Current weather in {city}: {random.choice(conditions)}, {temp}°C. Good conditions for sightseeing."
    print(f"   ↳ Returned: {result}")
    return result

def calculate_distance(
    from_location: Annotated[str, "Starting location"],
    to_location: Annotated[str, "Destination location"]
) -> str:
    """Calculate distance between two locations."""
    print(f"🔧 TOOL CALLED: calculate_distance('{from_location}', '{to_location}')")
    distance_km = round(random.uniform(2.0, 15.0), 1)
    travel_time = int(distance_km * 3)
    result = f"Distance from {from_location} to {to_location}: {distance_km} km, approximately {travel_time} minutes by car."
    print(f"   ↳ Returned: {result}")
    return result

# Concierge Agent Tools
def get_visitor_reviews(
    attraction_name: Annotated[str, "Name of the attraction"],
    city: Annotated[str, "City location"]
) -> str:
    """Get recent visitor reviews and ratings for an attraction."""
    print(f"🔧 TOOL CALLED: get_visitor_reviews('{attraction_name}', '{city}')")
    avg_rating = round(random.uniform(3.8, 4.9), 1)
    total_reviews = random.randint(1000, 10000)
    positive = random.choice(["amazing experience", "must-see", "well worth it"])
    concern = random.choice(["can be crowded", "expensive tickets", "long lines"])
    result = f"{attraction_name} in {city}: {avg_rating}/5.0 from {total_reviews} reviews. Most common positive: {positive}. Concern: {concern}."
    print(f"   ↳ Returned: {result[:80]}...")
    return result

def compare_attractions(
    attractions: Annotated[list[str], "List of attraction names to compare"],
    city: Annotated[str, "City location"]
) -> str:
    """Compare multiple attractions across various criteria."""
    print(f"🔧 TOOL CALLED: compare_attractions({attractions}, '{city}')")
    comparisons = []
    for attraction in attractions:
        rating = round(random.uniform(3.5, 5.0), 1)
        price = random.choice(["$", "$$", "$$$"])
        crowd = random.choice(["Low", "Medium", "High"])
        comparisons.append(f"{attraction}: Rating {rating}/5, Price {price}, Crowds: {crowd}")
    result = f"Comparison in {city}:\n" + "\n".join(comparisons)
    print(f"   ↳ Returned comparison data for {len(attractions)} attractions")
    return result

def get_ticket_prices(
    attraction_name: Annotated[str, "Name of the attraction"]
) -> str:
    """Get current ticket prices and booking information."""
    print(f"🔧 TOOL CALLED: get_ticket_prices('{attraction_name}')")
    adult = random.randint(15, 45)
    child = int(adult * 0.6)
    senior = int(adult * 0.8)
    result = f"{attraction_name} tickets: Adult ${adult}, Child ${child}, Senior ${senior}. Online booking: 10% discount."
    print(f"   ↳ Returned: {result}")
    return result

print("✅ Tool functions defined")
print("   Front Desk: 3 tools | Concierge: 3 tools")

✅ Tool functions defined
   Front Desk: 3 tools | Concierge: 3 tools


## Step 4: Configure Environment & Observability

Load environment variables and optionally configure Application Insights telemetry.

In [4]:
# Load environment variables
load_dotenv()

# Verify Azure AI Project endpoint
if not os.environ.get("AZURE_AI_PROJECT_ENDPOINT"):
    raise ValueError("❌ AZURE_AI_PROJECT_ENDPOINT environment variable is not set")

print(f"✅ Azure AI Project Endpoint: {os.environ['AZURE_AI_PROJECT_ENDPOINT']}")

# Optional: Configure Application Insights telemetry
if os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING"):
    try:
        from opentelemetry import trace
        from opentelemetry.sdk.trace import TracerProvider
        from opentelemetry.sdk.trace.export import BatchSpanProcessor
        from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
        
        provider = TracerProvider()
        trace.set_tracer_provider(provider)
        exporter = AzureMonitorTraceExporter(connection_string=os.environ["APPLICATIONINSIGHTS_CONNECTION_STRING"])
        provider.add_span_processor(BatchSpanProcessor(exporter))
        
        print("✅ Application Insights telemetry configured")
    except Exception as e:
        print(f"⚠️ Telemetry setup skipped: {e}")
else:
    print("ℹ️ Application Insights not configured (optional)")

✅ Azure AI Project Endpoint: https://camerjackson-9533-resource.services.ai.azure.com/api/projects/camerjackson-9533
✅ Application Insights telemetry configured


## Step 5: Retrieve Agents from Azure AI Foundry

Retrieve pre-configured agents from Azure AI Foundry and attach tool functions.

In [5]:
credential = AzureCliCredential()

async def get_foundry_agents():
    """Retrieve agents from Azure AI Foundry with tools."""
    async with AIProjectClient(
        endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
        credential=credential
    ) as project_client:
        provider = AzureAIProjectAgentProvider(project_client=project_client)
        
        # Define tools for each agent
        front_desk_tools = [get_attraction_hours, get_current_weather, calculate_distance]
        concierge_tools = [get_visitor_reviews, compare_attractions, get_ticket_prices]
        
        # Get agents with tools
        front_desk_agent = await provider.get_agent(name="frontdeskagent", tools=front_desk_tools)
        concierge_agent = await provider.get_agent(name="conciergeagent", tools=concierge_tools)
        
        print(f"✅ Retrieved Front Desk Agent: {front_desk_agent.name} | {len(front_desk_tools)} tools")
        print(f"✅ Retrieved Concierge Agent: {concierge_agent.name} | {len(concierge_tools)} tools")
        
        return front_desk_agent, concierge_agent

# Retrieve agents
front_desk_agent, concierge_agent = await get_foundry_agents()

✅ Retrieved Front Desk Agent: frontdeskagent | 3 tools
✅ Retrieved Concierge Agent: conciergeagent | 3 tools


## Step 6: Build Sequential Workflow

Create a sequential workflow where:
1. User input → Front Desk Agent (makes recommendation)
2. Front Desk output → Concierge Agent (reviews recommendation)
3. Final output contains both recommendation and expert review

In [6]:
# Build sequential workflow
workflow = (
    SequentialBuilder()
    .participants([front_desk_agent, concierge_agent])
    .build()
)

print("✅ Sequential workflow created")
print("   Flow: User → Front Desk → Concierge → Output")

✅ Sequential workflow created
   Flow: User → Front Desk → Concierge → Output


## Step 7: Define Display Functions

Helper functions to format and display agent responses.

In [7]:
def display_front_desk_section(data: AttractionRecommendation):
    """Display front desk recommendation."""
    display(HTML(f"""
    <div style='padding: 20px; background: #e3f2fd; border-radius: 8px; margin: 15px 0; border-left: 4px solid #2196f3;'>
        <h3 style='margin: 0 0 15px 0; color: #0d47a1; font-weight: bold;'>🏨 Front Desk Recommendation</h3>
        <h4 style='margin: 0 0 10px 0; color: #000; font-weight: bold;'>{data.attraction_name}</h4>
        <p style='color: #000; font-size: 15px; line-height: 1.6;'><strong>Category:</strong> {data.category}</p>
        <p style='color: #000; font-size: 15px; line-height: 1.6;'><strong>Description:</strong> {data.description}</p>
        <p style='color: #000; font-size: 15px; line-height: 1.6;'><strong>Why Recommended:</strong> {data.why_recommended}</p>
        <p style='color: #000; font-size: 15px; line-height: 1.6;'><strong>Duration:</strong> {data.recommended_duration}</p>
        <p style='color: #000; font-size: 15px; line-height: 1.6;'><strong>Best Time:</strong> {data.best_time_to_visit}</p>
    </div>
    """))

def display_concierge_section(data: AttractionReview):
    """Display concierge review."""
    star_rating = "⭐" * int(data.visitor_rating) + "☆" * (5 - int(data.visitor_rating))
    popularity_bar = "🟩" * data.popularity_score + "⬜" * (10 - data.popularity_score)
    pros_html = "".join([f"<li style='color: #2e7d32; font-size: 15px; font-weight: 500;'>✓ {pro}</li>" for pro in data.pros])
    cons_html = "".join([f"<li style='color: #c62828; font-size: 15px; font-weight: 500;'>✗ {con}</li>" for con in data.cons])
    
    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #e65100; font-weight: bold;'>🎩 Concierge Expert Review</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;'>
            <div style='background: rgba(255,152,0,0.2); padding: 15px; border-radius: 8px; border: 1px solid #ff9800;'>
                <h4 style='color: #000; font-weight: bold; margin: 0 0 10px 0;'>Popularity Score</h4>
                <div style='font-size: 28px; font-weight: bold; color: #000;'>{data.popularity_score}/10</div>
                <div style='font-size: 16px; margin-top: 8px;'>{popularity_bar}</div>
            </div>
            <div style='background: rgba(255,152,0,0.2); padding: 15px; border-radius: 8px; border: 1px solid #ff9800;'>
                <h4 style='color: #000; font-weight: bold; margin: 0 0 10px 0;'>Visitor Rating</h4>
                <div style='font-size: 28px; font-weight: bold; color: #000;'>{data.visitor_rating}/5.0</div>
                <div style='font-size: 18px; margin-top: 5px;'>{star_rating}</div>
            </div>
        </div>
        <p style='color: #000; font-size: 15px; line-height: 1.6;'><strong>Reasoning:</strong> {data.popularity_reasoning}</p>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px;'>
            <div><h4 style='color: #000; font-weight: bold;'>Pros:</h4><ul style='margin-top: 10px;'>{pros_html}</ul></div>
            <div><h4 style='color: #000; font-weight: bold;'>Cons:</h4><ul style='margin-top: 10px;'>{cons_html}</ul></div>
        </div>
        <p style='color: #000; font-size: 15px; line-height: 1.6; margin-top: 15px;'><strong>Recommendation:</strong> {data.concierge_recommendation}</p>
    </div>
    """))

def display_tool_calls(messages: list[ChatMessage]):
    """Display tool calls made by agents."""
    # Show available tools per agent
    tools_html = """
    <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-top: 10px;'>
        <div style='background: rgba(33,150,243,0.1); padding: 12px; border-radius: 6px; border: 1px solid #2196f3;'>
            <div style='color: #000; font-weight: 600; font-size: 14px; margin-bottom: 8px;'>🏨 Front Desk Agent Tools</div>
            <ul style='margin: 0; padding-left: 20px; color: #333; font-size: 13px;'>
                <li>get_attraction_hours()</li>
                <li>get_current_weather()</li>
                <li>calculate_distance()</li>
            </ul>
        </div>
        <div style='background: rgba(255,152,0,0.1); padding: 12px; border-radius: 6px; border: 1px solid #ff9800;'>
            <div style='color: #000; font-weight: 600; font-size: 14px; margin-bottom: 8px;'>🎩 Concierge Agent Tools</div>
            <ul style='margin: 0; padding-left: 20px; color: #333; font-size: 13px;'>
                <li>get_visitor_reviews()</li>
                <li>compare_attractions()</li>
                <li>get_ticket_prices()</li>
            </ul>
        </div>
    </div>
    """
    
    display(HTML(f"""
    <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; margin: 15px 0; border-left: 4px solid #4caf50;'>
        <h4 style='margin: 0 0 10px 0; color: #2e7d32; font-weight: bold;'>🛠️ Tool Usage</h4>
        <p style='color: #000; font-size: 14px; margin: 0 0 10px 0;'>Agents have access to the following tools to gather real-time information:</p>
        {tools_html}
        <p style='color: #666; font-size: 13px; margin: 15px 0 0 0; font-style: italic;'>Tool calls and outputs are handled internally by the agents and integrated into their responses.</p>
    </div>
    """))

print("✅ Display functions defined")

✅ Display functions defined


## Step 8: Run the Workflow

Execute the sequential workflow with a city and display formatted results.

In [8]:
async def display_attraction_recommendation(city: str):
    """Run workflow and display results."""
    
    # Rebuild workflow to get fresh state
    fresh_workflow = (
        SequentialBuilder()
        .participants([front_desk_agent, concierge_agent])
        .build()
    )
    
    # Create trace span for telemetry (if configured)
    with get_tracer().start_as_current_span(f"Attraction-Recommendation-{city}", kind=SpanKind.CLIENT) as span:
        display(HTML(f"""
        <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; border: 1px solid #ff9800;'>
            <h3 style='color: #000; font-weight: bold; margin: 0 0 10px 0;'>🔄 Processing Recommendation for {city}</h3>
            <p style='color: #666; font-size: 13px; font-style: italic; margin: 0;'>Telemetry data is being sent to Application Insights (if configured)</p>
        </div>
        """))
        
        # Run workflow
        events = await fresh_workflow.run(f"I want to visit an attraction in {city}. Please check the current weather, opening hours, visitor reviews, and ticket prices to help me decide.")
    
    outputs = events.get_outputs()
    
    if outputs:
        messages: list[ChatMessage] = outputs[0]
        
        # Extract agent responses
        front_desk_response = None
        concierge_response = None
        
        for msg in messages:
            if msg.author_name == "frontdeskagent":
                front_desk_response = msg.text
            elif msg.author_name == "conciergeagent":
                concierge_response = msg.text
        
        # Display header
        display(HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #2e7d32 0%, #66bb6a 100%); 
                    color: white; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 6px rgba(0,0,0,0.2);'>
            <h2 style='margin: 0 0 10px 0; font-weight: bold; text-shadow: 1px 1px 2px rgba(0,0,0,0.3);'>Attraction Recommendation for {city}</h2>
            <p style='margin: 0; font-size: 16px; opacity: 0.95;'>Generated by sequential agent workflow</p>
        </div>
        """))
        
        # Display tool calls
        display_tool_calls(messages)
        
        # Parse and display responses
        if front_desk_response:
            try:
                recommendation = AttractionRecommendation.model_validate_json(front_desk_response)
                display_front_desk_section(recommendation)
            except Exception as e:
                display(HTML(f"<div style='color: #c62828; background: #ffebee; padding: 15px; border-radius: 8px; border-left: 4px solid #c62828; font-size: 15px; font-weight: 500;'>❌ Error parsing front desk response: {e}</div>"))
        
        if concierge_response:
            try:
                review = AttractionReview.model_validate_json(concierge_response)
                display_concierge_section(review)
            except Exception as e:
                display(HTML(f"<div style='color: #c62828; background: #ffebee; padding: 15px; border-radius: 8px; border-left: 4px solid #c62828; font-size: 15px; font-weight: 500;'>❌ Error parsing concierge response: {e}</div>"))

# Test the workflow
await display_attraction_recommendation("Barcelona")

## Step 8a: View Workflow Traces in Azure Portal (Optional)

After running Step 8, you can view the telemetry data in Azure Portal's Application Insights.

### How to View Traces:

1. **Navigate to Application Insights:**
   - Go to the [Azure Portal](https://portal.azure.com)
   - Find your Application Insights resource

2. **Open Search:**
   - In the left sidebar, locate the **Investigate** section
   - Click on **Search**

3. **View Recent Data:**
   - Click the **"See all data in the last 24 hours"** button at the top
   - You'll see all telemetry events from your agent workflow runs

4. **Find Your Workflow:**
   - Look for entries closests to the timestamp you executed the workflow.
   - Each workflow execution appears as a separate operation
   - Sort by timestamp to find the most recent runs
   - Click on any entry to see detailed telemetry including duration, dependencies, and custom properties

**Tip:** The Search view shows all telemetry types (requests, dependencies, traces) in a unified timeline, making it easy to troubleshoot and understand the complete workflow execution.

## Step 9: Analyze Workflow (Optional)

Examine the message flow between agents to understand the sequential orchestration.

**Enhanced Telemetry:** This step includes custom logging that sends detailed agent conversation messages to Application Insights. Messages appear as events under the operation name "Agent-Conversation-{city}".

In [9]:
def log_agent_messages_to_app_insights(messages: list[ChatMessage], city: str):
    """Log agent conversation messages to Application Insights as traces."""
    import logging
    from opentelemetry import trace
    from opentelemetry.sdk.trace import TracerProvider
    
    # Get logger that will output to App Insights
    logger = logging.getLogger(__name__)
    tracer = get_tracer()
    
    with tracer.start_as_current_span(f"Agent-Conversation-{city}", kind=SpanKind.INTERNAL) as parent_span:
        # Add span attributes with summary
        parent_span.set_attribute("city", city)
        parent_span.set_attribute("message_count", len(messages))
        parent_span.set_attribute("agent_count", 2)
        
        # Log each message as a separate child span with custom dimensions
        for i, msg in enumerate(messages):
            author = msg.author_name or "user"
            content_preview = msg.text  # Full content, no truncation
            
            with tracer.start_as_current_span(f"Message-{i+1}-{author}", kind=SpanKind.INTERNAL) as msg_span:
                msg_span.set_attribute("message_number", i + 1)
                msg_span.set_attribute("author", author)
                msg_span.set_attribute("content_length", len(msg.text))
                msg_span.set_attribute("content", content_preview)
                msg_span.set_attribute("has_full_content", True)
                msg_span.set_attribute("city", city)
        
        print(f"✅ Logged {len(messages)} messages to Application Insights")


async def analyze_sequential_flow(city: str):
    """Analyze the sequential flow between agents."""
    
    # Rebuild workflow to get fresh state
    fresh_workflow = (
        SequentialBuilder()
        .participants([front_desk_agent, concierge_agent])
        .build()
    )
    
    with get_tracer().start_as_current_span(f"Flow-Analysis-{city}", kind=SpanKind.CLIENT) as span:
        display(HTML(f"""
        <div style='padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 8px; border: 1px solid #9c27b0;'>
            <h3 style='color: #000; font-weight: bold; margin: 0 0 10px 0;'>🔍 Sequential Flow Analysis for {city}</h3>
            <p style='color: #666; font-size: 13px; font-style: italic; margin: 0;'>Analyzing agent communication flow and telemetry</p>
        </div>
        """))
        
        events = await fresh_workflow.run(f"I want to visit an attraction in {city}. Please check the current weather, opening hours, visitor reviews, and ticket prices to help me decide.")
    
    outputs = events.get_outputs()
    
    if outputs:
        messages: list[ChatMessage] = outputs[0]
        
        # Log messages to Application Insights
        log_agent_messages_to_app_insights(messages, city)
        
        display(HTML("<h3 style='color: #000; font-weight: bold;'>Conversation Flow</h3>"))
        
        for i, msg in enumerate(messages, 1):
            role_name = {
                "user": "👤 User",
                "frontdeskagent": "🏨 Front Desk Agent",
                "conciergeagent": "🎩 Concierge Agent"
            }.get(msg.author_name or "user", "Unknown")
            
            preview = msg.text[:200] + "..." if len(msg.text) > 200 else msg.text
            
            display(HTML(f"""
            <div style='padding: 15px; margin: 10px 0; background: white; 
                        border-left: 4px solid #666; border-radius: 4px; border: 1px solid #ddd;'>
                <strong style='color: #000; font-size: 16px;'>Step {i}: {role_name}</strong>
                <p style='font-size: 15px; color: #333; margin-top: 8px; line-height: 1.5;'>{preview}</p>
            </div>
            """))
        
        display(HTML(f"""
        <div style='padding: 20px; background: linear-gradient(135deg, #7b1fa2 0%, #9c27b0 100%); 
                    color: white; border-radius: 8px; margin: 20px 0; box-shadow: 0 4px 6px rgba(0,0,0,0.2);'>
            <h3 style='margin: 0 0 15px 0; font-weight: bold; text-shadow: 1px 1px 2px rgba(0,0,0,0.3);'>Flow Summary</h3>
            <ul style='font-size: 15px; line-height: 1.8;'>
                <li><strong>Total Messages:</strong> {len(messages)}</li>
                <li><strong>Agents:</strong> 2 (Front Desk + Concierge)</li>
                <li><strong>Pattern:</strong> Linear sequential (User → Agent 1 → Agent 2)</li>
                <li><strong>Quality:</strong> Enhanced through expert review</li>
            </ul>
        </div>
        """))

# Optional: Analyze flow
await analyze_sequential_flow("Barcelona")

✅ Logged 3 messages to Application Insights


## Step 9a: Query Agent Messages from Application Insights (Optional)

After running Step 9, wait 1-5 minutes for telemetry ingestion, then query to view the logged messages.

**Note:** Full message content is captured (no truncation). Application Insights supports up to 64KB per custom dimension.

### Query Options

Two queries are available depending on what you want to see:

#### Query 1: View Individual Messages (Full Conversation Content)

Shows each message in the conversation with full content:

```kusto
dependencies
| where name startswith 'Message-'
| extend msg_num = toint(customDimensions.message_number), author = tostring(customDimensions.author), content = tostring(customDimensions.content)
| project msg_num, author, content
| order by msg_num asc
```

**Use this when:** You want to see what was said by each agent (user request, frontdeskagent JSON, conciergeagent JSON)

#### Query 2: View Conversation Summary (Metadata Only)

Shows high-level information about the conversation:

```kusto
dependencies
| where name == "Agent-Conversation-Barcelona"
| extend city = tostring(customDimensions.city)
| extend message_count = toint(customDimensions.message_count)
| extend agent_count = toint(customDimensions.agent_count)
| project timestamp, city, message_count, agent_count, duration
| order by timestamp desc
| take 1
```

**Use this when:** You want to see when the conversation ran, which city, how many messages were exchanged, and how long it took

---

**How to run these queries:**

**Option A - In Azure Portal:**
1. Go to your Application Insights resource
2. Click "Logs" in the left menu
3. Paste either query above
4. Click "Run"

**Option B - In Azure CLI (terminal):**
See the examples in the code cell below with the `!az monitor app-insights query` commands

In [ ]:
# Query Application Insights for logged agent messages
# Two query options:

# Replace <YOUR-APP-INSIGHTS-ID> with your Application Insights App ID
# You can find this in Azure Portal -> Application Insights -> Properties -> Application ID

# Option 1: Get individual messages with full content (user, frontdeskagent, conciergeagent)
# !az monitor app-insights query --app <YOUR-APP-INSIGHTS-ID> --analytics-query "dependencies | where name startswith 'Message-' | extend msg_num = toint(customDimensions.message_number), author = tostring(customDimensions.author), content = tostring(customDimensions.content) | project msg_num, author, content | order by msg_num asc"

# Option 2: Get conversation summary (city, message count, duration)
# !az monitor app-insights query --app <YOUR-APP-INSIGHTS-ID> --analytics-query "dependencies | where name == 'Agent-Conversation-Barcelona' | extend city = tostring(customDimensions.city) | extend message_count = toint(customDimensions.message_count) | extend agent_count = toint(customDimensions.agent_count) | project timestamp, city, message_count, agent_count, duration | order by timestamp desc | take 1"

## Try Different Cities!

Test the workflow with various cities to see how agents adapt their recommendations.

In [ ]:
cities_to_try = ["Paris", "Tokyo", "New York", "Barcelona", "Rome"]

# Option 1: Test a single city
# Uncomment to test:
# await display_attraction_recommendation("Paris")

# Option 2: Loop through all cities
# Uncomment to test all cities:
# for city in cities_to_try:
#     print(f"\n{'='*80}\n🌍 Testing: {city}\n{'='*80}")
#     await display_attraction_recommendation(city)

---

## 🎉 Congratulations!

You've completed the Sequential Agent Workflow with Code! You've built a production-ready multi-agent system using Microsoft Agent Framework with full observability.

### What You Learned

✅ **Sequential Agent Workflows** - Orchestrated multi-agent systems with code  
✅ **Agent Handoffs** - Passed information between specialized agents  
✅ **Structured Outputs** - Used Pydantic models for type-safe data  
✅ **Custom Tool Functions** - Enhanced agents with additional capabilities  
✅ **OpenTelemetry Tracing** - Added custom spans for workflow monitoring  
✅ **Application Insights Integration** - Tracked performance and debugged issues  
✅ **Production Patterns** - Implemented real-world agent orchestration

### Resources

**Documentation:**
- [Microsoft Agent Framework](https://github.com/microsoft/agent-framework) - Official framework documentation
- [Azure AI Foundry](https://learn.microsoft.com/azure/ai-foundry/) - Platform documentation
- [Application Insights](https://learn.microsoft.com/azure/azure-monitor/app/app-insights-overview) - Monitoring documentation

**Related Lessons:**
- [Lesson 10: AI Agents in Production](../../10-ai-agents-production/README.md) - Production deployment strategies
- [Lesson 08: Multi-Agent Systems](../../08-multi-agent/README.md) - Advanced multi-agent patterns
- [Lesson 14: Microsoft Agent Framework](../../14-microsoft-agent-framework/README.md) - Framework deep dive

**Community:**
- [Azure AI Foundry Discord](https://aka.ms/ai-agents/discord) - Get help and share learnings
- [Workshop Issues](https://github.com/microsoft/ai-agents-for-beginners/issues) - Report issues or suggest improvements